# 1-Shot RLVR with 4-bit Quantization on T4 GPU


## Cell 1: Check GPU

In [1]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU name       : {torch.cuda.get_device_name(0)}')
print(f'Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Sun May 17 08:05:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

## Cell 2: Install Dependencies

In [3]:
# ── Install all required packages ──────────────────────────────────────────
# Using specific versions to avoid compatibility issues on Colab
!pip install -q \
    peft==0.14.0 \
    bitsandbytes==0.45.0 \
    accelerate==1.2.1 \
    datasets==3.2.0 \
    math-verify \
    sentencepiece

print('✅ All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
✅ All packages installed


In [4]:
!pip install -q --upgrade --force-reinstall \
    trl==0.15.2 \
    transformers==4.46.3 \
    accelerate==1.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 59.2 MB/s eta 0:00:00
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 54.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 100.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 780.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 15.1 MB/s eta 0:00:00
    

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import re
import gc
import json
import torch
import numpy as np
from datetime import datetime
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import GRPOConfig, GRPOTrainer

# Suppress minor warnings
import warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Imports successful')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {torch.cuda.get_device_name(0)}')

✅ Imports successful
   PyTorch  : 2.5.1+cu121
   Device   : Tesla T4


## Cell 4: Configuration
All hyperparameters in one place — tweak here if you get OOM errors.

In [2]:
import os
import re
import gc
import json
import torch
import numpy as np
from datetime import datetime
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import GRPOConfig, GRPOTrainer

# Suppress minor warnings
import warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Imports successful')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {torch.cuda.get_device_name(0)}')

# ── Configuration ──────────────────────────────────────────────────────────
# NOTE: If you get OOM errors, reduce MAX_NEW_TOKENS or NUM_GENERATIONS first

CFG = dict(
    # Model
    model_name        = 'Qwen/Qwen2.5-Math-1.5B',   # same base model as paper

    # Quantization
    load_in_4bit      = True,
    quant_type        = 'nf4',                        # NormalFloat4 — best for LLMs
    compute_dtype     = torch.float16,
    double_quant      = True,                         # saves extra ~0.4 GB

    # LoRA  (only train a small adapter, not full weights)
    lora_r            = 16,
    lora_alpha        = 32,
    lora_dropout      = 0.05,
    # which layers to adapt (attention + feed-forward)
    lora_target       = ['q_proj','k_proj','v_proj','o_proj',
                         'gate_proj','up_proj','down_proj'],

    # GRPO Training  (heavily reduced vs paper to fit T4)
    # Paper used: batch=128, rollouts=8, response_len=3072, steps=2000
    per_device_batch  = 2,     # T4 constraint
    grad_accum        = 1,     # effective batch = 4 (paper used 128)
    num_generations   = 2,     # rollouts per prompt (paper used 8)
    max_new_tokens    = 128,   # response length (paper used 3072) -- REDUCED FROM 128
    max_prompt_len    = 256,  # REDUCED FROM 512
    num_train_steps   = 200,   # paper used 2000; 200 enough to see the effect
    learning_rate     = 5e-6,
    temperature       = 0.6,   # same as paper
    save_steps        = 50,
    logging_steps     = 10,

    # Dataset replication trick from paper:
    # paper duplicated single example to fill batch of 128
    dataset_repeat    = 64,    # repeat π1 to fill dataset

    # Output
    output_dir        = './oneshot_rlvr_output',
)

print('✅ Configuration set')
print(f"   Model          : {CFG['model_name']}")
print(f"   Quantization   : 4-bit {CFG['quant_type']}")
print(f"   LoRA rank      : {CFG['lora_r']}")
print(f"   Generations    : {CFG['num_generations']} per prompt")
print(f"   Max new tokens : {CFG['max_new_tokens']}")
print(f"   Train steps    : {CFG['num_train_steps']}")

✅ Imports successful
   PyTorch  : 2.5.1+cu121
   Device   : Tesla T4
✅ Configuration set
   Model          : Qwen/Qwen2.5-Math-1.5B
   Quantization   : 4-bit nf4
   LoRA rank      : 16
   Generations    : 2 per prompt
   Max new tokens : 128
   Train steps    : 200


## Cell 5: The Training Example (π₁ from Paper)
This is the **exact same** example the paper used — the wind pressure algebra problem.

In [3]:
# ── π₁ — The single training example from the paper (Table 2) ─────────────
# Paper found this single example raises MATH500 from 36% → 73.6%

PI_1_PROMPT = (
    "The pressure P exerted by wind on a sail varies jointly as the area A of the sail "
    "and the cube of the wind's velocity V. "
    "When the velocity is 8 miles per hour, the pressure on a sail of 2 square feet is 4 pounds. "
    "Find the wind velocity when the pressure on 4 square feet of sail is 32 pounds. "
    "Let's think step by step and output the final answer within \\boxed{}."
)
PI_1_ANSWER = "12.8"   # ground truth label from paper

# We also include π₁₃ (geometry example from paper, Table 21)
# to test 2-shot as paper did
PI_13_PROMPT = (
    "Given that circle C passes through points P(0,-4), Q(2,0), and R(3,-1). "
    "(1) Find the equation of circle C. "
    "(2) If the line l: mx+y-1=0 intersects circle C at points A and B, "
    "and |AB|=4, find the value of m. "
    "Let's think step by step and output the final answer within \\boxed{}."
)
PI_13_ANSWER = "4/3"

print('✅ Training examples loaded')
print(f'   π₁  answer: {PI_1_ANSWER}')
print(f'   π₁₃ answer: {PI_13_ANSWER}')
print()
print('π₁ prompt preview:')
print(PI_1_PROMPT[:120], '...')

✅ Training examples loaded
   π₁  answer: 12.8
   π₁₃ answer: 4/3

π₁ prompt preview:
The pressure P exerted by wind on a sail varies jointly as the area A of the sail and the cube of the wind's velocity V. ...


## Cell 6: Build Dataset
Paper trick: duplicate the single example to fill the training batch.

In [4]:
# ── Build 1-shot dataset (paper Section 3.1 trick) ─────────────────────────
# Paper: 'we duplicate the selected example until reaching 128 samples'
# We do the same but with smaller batch size

def build_dataset(prompt: str, answer: str, repeat: int) -> Dataset:
    """Repeat a single example to fill the dataset (same trick as paper)."""
    return Dataset.from_dict({
        'prompt' : [prompt] * repeat,
        'answer' : [answer] * repeat,
    })

# 1-shot dataset using π₁ only
train_dataset_1shot = build_dataset(
    PI_1_PROMPT, PI_1_ANSWER, CFG['dataset_repeat']
)

# 2-shot dataset using π₁ + π₁₃ (alternating)
train_dataset_2shot = Dataset.from_dict({
    'prompt' : ([PI_1_PROMPT, PI_13_PROMPT] * (CFG['dataset_repeat'] // 2)),
    'answer' : ([PI_1_ANSWER, PI_13_ANSWER] * (CFG['dataset_repeat'] // 2)),
})

print(f'✅ Datasets built')
print(f'   1-shot dataset size : {len(train_dataset_1shot)}')
print(f'   2-shot dataset size : {len(train_dataset_2shot)}')
print(f'   (All rows are repeated copies of 1 or 2 examples — same as paper)')

✅ Datasets built
   1-shot dataset size : 64
   2-shot dataset size : 64
   (All rows are repeated copies of 1 or 2 examples — same as paper)


## Cell 7: Reward Function
Binary reward: **1** if answer matches, **0** otherwise — same as paper.

In [19]:
# ── Reward function — binary 0/1 exactly like the paper ───────────────────

def extract_boxed_answer(text: str) -> str:
    """Extract content from \\boxed{...} in model output."""
    # Handle nested braces e.g. \boxed{\frac{4}{3}}
    pattern = r'\\boxed\{([^{}]*)\}'
    matches = re.findall(pattern, text)
    if matches:
        return matches[-1].strip()   # take last boxed answer

    # Fallback: look for boxed with nested braces
    start = text.rfind(r'\boxed{')
    if start == -1:
        return ''
    depth, i = 0, start + len(r'\boxed{')
    content_start = i
    while i < len(text):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            if depth == 0:
                return text[content_start:i].strip()
            depth -= 1
        i += 1
    return ''


def normalize_answer(ans: str) -> str:
    """Normalize answer string for comparison."""
    ans = ans.strip().lower()
    # Remove spaces and common formatting
    ans = ans.replace(' ', '').replace(',', '')
    # Normalize common fraction formats
    ans = ans.replace('\\frac{4}{3}', '4/3').replace('\\frac', '')
    # Try numeric comparison
    try:
        return str(round(float(ans), 4))
    except:
        return ans


def math_reward_fn(completions, answer, **kwargs):
    """
    Binary reward function — exactly as described in paper Section 2.
    Returns 1.0 if model answer matches ground truth, else 0.0

    Args:
        completions: list of model-generated strings
        answer     : list of ground truth answer strings
    Returns:
        list of float rewards (0.0 or 1.0)
    """
    rewards = []
    for completion, gt in zip(completions, answer):
        predicted = extract_boxed_answer(completion)
        # Try exact match first
        if normalize_answer(predicted) == normalize_answer(gt):
            rewards.append(1.0)
            continue
        # Try numeric tolerance match (for floating point answers)
        try:
            pred_val = float(predicted.replace(',', ''))
            gt_val   = float(gt.replace(',', ''))
            # rewards.append(1.0 if abs(pred_val - gt_val) < 0.01 else 0.0)
            rewards.append(1.0 if abs(pred_val - gt_val) < 0.15 else 0.0)
        except:
            rewards.append(0.0)
    return rewards


# Quick test
test_completions = [
    'The answer is \\boxed{12.8} miles per hour.',
    'So the velocity is \\boxed{12.7}.',
    'I get \\boxed{10}.',
]
test_answers = ['12.8', '12.8', '12.8']
test_rewards = math_reward_fn(test_completions, test_answers)

print('✅ Reward function test:')
for comp, rew in zip(test_completions, test_rewards):
    extracted = extract_boxed_answer(comp)
    print(f'   Extracted: "{extracted}" → reward = {rew}')

✅ Reward function test:
   Extracted: "12.8" → reward = 1.0
   Extracted: "12.7" → reward = 1.0
   Extracted: "10" → reward = 0.0


## Cell 8: Load Model in 4-bit
This is the **key difference** from the paper — 4-bit QLoRA instead of full fp16.

In [20]:
# ── Load model in 4-bit quantization ──────────────────────────────────────
# NOVEL: Paper used full fp16. We use 4-bit to fit T4.

def get_gpu_memory_gb():
    return torch.cuda.memory_allocated() / 1e9

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_name'],
    trust_remote_code=True,
    padding_side='left',   # important for batch generation
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = CFG['load_in_4bit'],
    bnb_4bit_quant_type       = CFG['quant_type'],       # nf4
    bnb_4bit_compute_dtype    = CFG['compute_dtype'],    # fp16
    bnb_4bit_use_double_quant = CFG['double_quant'],     # saves ~0.4GB more
)

print('Loading model in 4-bit (this may take 2-3 minutes)...')
mem_before = get_gpu_memory_gb()

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_name'],
    quantization_config = bnb_config,
    device_map          = 'auto',
    trust_remote_code   = True,
    torch_dtype         = torch.float16,
)

mem_after = get_gpu_memory_gb()
print(f'✅ Model loaded')
print(f'   GPU memory used by model : {mem_after - mem_before:.2f} GB')
print(f'   Total GPU memory used    : {mem_after:.2f} GB')
print(f'   GPU memory free          : {(torch.cuda.get_device_properties(0).total_memory / 1e9) - mem_after:.2f} GB')

Loading tokenizer...
Loading model in 4-bit (this may take 2-3 minutes)...
✅ Model loaded
   GPU memory used by model : 1.15 GB
   Total GPU memory used    : 2.88 GB
   GPU memory free          : 12.76 GB


## Cell 9: Baseline Evaluation (Before Training)
Record base model performance — this is our starting point to compare against.

In [17]:
# # ── Evaluate base model BEFORE training ───────────────────────────────────
# # Paper reports base model: MATH500 = 36.0%
# # We test on a small subset to save time

# # Small test set — mix of easy and medium math problems
# # TEST_PROBLEMS = [
# #     # (problem, answer)
# #     ("What is 15% of 80? Let's think step by step and output the final answer within \\boxed{}.",
# #      "12"),
# #     ("Solve for x: 2x + 5 = 13. Let's think step by step and output the final answer within \\boxed{}.",
# #      "4"),
# #     ("A rectangle has length 8 and width 5. What is its area? Let's think step by step and output the final answer within \\boxed{}.",
# #      "40"),
# #     ("If f(x) = x^2 + 2x + 1, what is f(3)? Let's think step by step and output the final answer within \\boxed{}.",
# #      "16"),
# #     ("What is the sum of the first 10 natural numbers? Let's think step by step and output the final answer within \\boxed{}.",
# #      "55"),
# #     ("Simplify: (x^2 - 4) / (x - 2). Let's think step by step and output the final answer within \\boxed{}.",
# #      "x+2"),
# #     ("A train travels 120 miles in 2 hours. What is its speed in mph? Let's think step by step and output the final answer within \\boxed{}.",
# #      "60"),
# #     ("What is the derivative of x^3 + 2x? Let's think step by step and output the final answer within \\boxed{}.",
# #      "3x^2+2"),
# # ]

# TEST_PROBLEMS = [
#     ("Find the sum of all positive integers n such that n^2 - 19n + 99 is a perfect square. Let's think step by step and output the final answer within \\boxed{}.", "38"),
#     ("How many integers between 1 and 200 are divisible by both 3 and 5 but not by 7? Let's think step by step and output the final answer within \\boxed{}.", "12"),
#     ("A circle has area 49π. What is its circumference? Let's think step by step and output the final answer within \\boxed{}.", "14π"),
#     ("If log base 2 of x = 3, what is log base 4 of x? Let's think step by step and output the final answer within \\boxed{}.", "3/2"),
#     ("What is the units digit of 7^2025? Let's think step by step and output the final answer within \\boxed{}.", "7"),
#     ("Two dice are rolled. What is the probability both show the same number? Let's think step by step and output the final answer within \\boxed{}.", "1/6"),
#     ("Find the number of real solutions to x^4 - 5x^2 + 4 = 0. Let's think step by step and output the final answer within \\boxed{}.", "4"),
#     ("A geometric sequence has first term 3 and common ratio 2. What is the 8th term? Let's think step by step and output the final answer within \\boxed{}.", "384"),
# ]


# def evaluate_model(model, tokenizer, test_problems, desc='Evaluation'):
#     """Run greedy evaluation on test problems."""
#     model.eval()
#     correct, total = 0, 0
#     results = []

#     with torch.no_grad():
#         for problem, gt_answer in test_problems:
#             inputs = tokenizer(
#                 problem,
#                 return_tensors='pt',
#                 truncation=True,
#                 max_length=CFG['max_prompt_len'],
#             ).to(model.device)

#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens = 256,
#                 do_sample      = False,   # greedy for evaluation
#                 pad_token_id   = tokenizer.eos_token_id,
#             )

#             # Decode only the new tokens
#             new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
#             response   = tokenizer.decode(new_tokens, skip_special_tokens=True)

#             predicted  = extract_boxed_answer(response)
#             rewards    = math_reward_fn([response], [gt_answer])
#             is_correct = rewards[0] == 1.0

#             if is_correct:
#                 correct += 1
#             total += 1

#             results.append({
#                 'problem'  : problem[:60] + '...',
#                 'gt'       : gt_answer,
#                 'predicted': predicted,
#                 'correct'  : is_correct,
#             })

#     accuracy = correct / total * 100
#     print(f'\n{desc} Results:')
#     print(f'  Accuracy: {correct}/{total} = {accuracy:.1f}%')
#     for r in results:
#         status = '✅' if r['correct'] else '❌'
#         print(f'  {status} GT={r["gt"]:10s} | Pred={r["predicted"]:10s} | {r["problem"]}')
#     return accuracy


# print('Running baseline evaluation (before training)...')
# baseline_accuracy = evaluate_model(
#     model, tokenizer, TEST_PROBLEMS, 'BASELINE (Before Training)'
# )
# print(f'\n📊 Baseline accuracy: {baseline_accuracy:.1f}%')
# print('(Paper baseline on MATH500 was 36.0%)')


# ── MATH500 Evaluation — matches paper exactly ────────────────────────────
# Paper uses the official Qwen2.5-Math evaluation pipeline on MATH500
# Dataset: HuggingFaceH4/MATH-500 (500 problems from MATH test set)
# Same 7 subject categories as paper Table 3

from datasets import load_dataset
import re, torch
from tqdm import tqdm

# ── Load MATH500 ───────────────────────────────────────────────────────────
print("Loading MATH500 dataset...")
math500 = load_dataset("HuggingFaceH4/MATH-500", split="test")
print(f"Loaded {len(math500)} problems")
print(f"Subjects: {sorted(set(math500['subject']))}")

# ── Prompt template — same as Qwen2.5-Math paper ──────────────────────────
QWEN_MATH_TEMPLATE = (
    "Please reason step by step, and put your final answer within \\boxed{{}}.\n\n"
    "Problem: {problem}"
)

# ── Answer normalisation (handles LaTeX fractions, decimals, etc.) ─────────
def normalise_math_answer(ans: str) -> str:
    """
    Normalise a LaTeX/text math answer for comparison.
    Handles common forms the model and dataset both produce.
    """
    if not ans:
        return ""
    ans = ans.strip()

    # Remove surrounding $...$
    ans = re.sub(r'^\$+|\$+$', '', ans).strip()

    # Normalise LaTeX fractions: \frac{a}{b} → a/b
    ans = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', ans)

    # Remove LaTeX formatting
    for cmd in [r'\\left', r'\\right', r'\\,', r'\\!', r'\\ ', r'\\text\{[^}]*\}']:
        ans = re.sub(cmd, '', ans)
    ans = re.sub(r'\\(sqrt|cdot|times|pm|approx)', r'\\\1', ans)

    # Normalise spaces and case
    ans = ans.replace(' ', '').lower()

    # Try numeric normalisation
    try:
        val = float(ans.replace(',', ''))
        # Round to 4 dp to handle floating point drift
        return str(round(val, 4))
    except ValueError:
        return ans


def extract_boxed_answer(text: str) -> str:
    """Extract the last \\boxed{...} content, handling nested braces."""
    # Simple non-nested first
    matches = re.findall(r'\\boxed\{([^{}]*)\}', text)
    if matches:
        return matches[-1].strip()
    # Nested brace fallback
    start = text.rfind(r'\boxed{')
    if start == -1:
        return ''
    depth, i = 0, start + len(r'\boxed{')
    content_start = i
    while i < len(text):
        if   text[i] == '{': depth += 1
        elif text[i] == '}':
            if depth == 0:
                return text[content_start:i].strip()
            depth -= 1
        i += 1
    return ''


def answers_match(predicted: str, ground_truth: str) -> bool:
    """Check if two answers are equivalent."""
    pred_norm = normalise_math_answer(predicted)
    gt_norm   = normalise_math_answer(ground_truth)

    if pred_norm == gt_norm:
        return True

    # Numeric tolerance (handles 12.7 vs 12.8 case from paper footnote 4)
    try:
        return abs(float(pred_norm) - float(gt_norm)) < 0.15
    except ValueError:
        return False


# ── Main evaluation function ───────────────────────────────────────────────
def evaluate_math500(model, tokenizer, dataset=math500,
                     max_new_tokens=256, desc="MATH500 Evaluation",
                     batch_size=1):
    """
    Evaluate model on full MATH500 — matches paper setup.

    Returns:
        overall_acc  : float  (0-100)
        subject_accs : dict   subject → accuracy
        results      : list   per-problem dicts
    """
    model.eval()
    results = []

    subject_correct = {}
    subject_total   = {}

    print(f"\n{'='*60}")
    print(f"{desc}")
    print(f"Evaluating {len(dataset)} problems | max_new_tokens={max_new_tokens}")
    print(f"{'='*60}\n")

    for i, item in enumerate(tqdm(dataset, desc=desc)):
        problem  = item['problem']
        gt       = item['answer']        # ground truth answer string
        subject  = item['subject']
        level    = item.get('level', '?')

        prompt = QWEN_MATH_TEMPLATE.format(problem=problem)

        inputs = tokenizer(
            prompt,
            return_tensors  = 'pt',
            truncation      = True,
            max_length      = 512,      # keep prompt intact
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,          # greedy — same as paper (temp=0)
                pad_token_id   = tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
        predicted  = extract_boxed_answer(response)
        correct    = answers_match(predicted, gt)

        # Track per-subject
        subject_correct[subject] = subject_correct.get(subject, 0) + int(correct)
        subject_total[subject]   = subject_total.get(subject, 0) + 1

        results.append({
            'problem'  : problem,
            'gt'       : gt,
            'predicted': predicted,
            'correct'  : correct,
            'subject'  : subject,
            'level'    : level,
        })

    # ── Summary ────────────────────────────────────────────────────────────
    total_correct = sum(r['correct'] for r in results)
    overall_acc   = total_correct / len(results) * 100

    subject_accs  = {
        subj: subject_correct[subj] / subject_total[subj] * 100
        for subj in sorted(subject_total)
    }

    print(f"\n{'='*60}")
    print(f"RESULTS: {desc}")
    print(f"{'='*60}")
    print(f"  Overall accuracy : {total_correct}/{len(results)} = {overall_acc:.1f}%")
    print(f"  (Paper baseline  : 36.0%  |  Paper post-1shot: 73.6%)")
    print()

    # Paper Table 3 subject abbreviations
    abbrev = {
        'Algebra'               : 'Alg.',
        'Counting & Probability': 'C.P.',
        'Geometry'              : 'Geo.',
        'Intermediate Algebra'  : 'I.Alg.',
        'Number Theory'         : 'N.T.',
        'Prealgebra'            : 'Prealg.',
        'Precalculus'           : 'Precal.',
    }
    print(f"  {'Subject':<28} {'Ours':>8}   {'Paper base':>10}")
    print(f"  {'-'*50}")
    paper_base = {            # from paper Table 3, Base row
        'Algebra'               : 37.1,
        'Counting & Probability': 31.6,
        'Geometry'              : 39.0,
        'Intermediate Algebra'  : 43.3,
        'Number Theory'         : 24.2,
        'Prealgebra'            : 36.6,
        'Precalculus'           : 33.9,
    }
    for subj, acc in subject_accs.items():
        ab  = abbrev.get(subj, subj[:7])
        ref = paper_base.get(subj, 0.0)
        print(f"  {ab:<28} {acc:>7.1f}%   {ref:>9.1f}%")

    print(f"{'='*60}\n")
    return overall_acc, subject_accs, results


# ── Run baseline evaluation ────────────────────────────────────────────────
# NOTE: 500 problems × ~5 sec each on T4 ≈ 40-45 minutes
# If you want a quick check first, use: dataset=math500.select(range(50))
print("Starting MATH500 baseline evaluation...")
print("Tip: use math500.select(range(50)) for a quick 50-problem sanity check first\n")

baseline_acc, baseline_subject_accs, baseline_results = evaluate_math500(
    model, tokenizer,
    # dataset       = math500,
    dataset = math500.select(range(10)),
    max_new_tokens= 256,
    desc          = "BASELINE — Before 1-Shot RLVR",
)

baseline_accuracy = baseline_acc   # keep for later cells
print(f"📊 Baseline MATH500 accuracy: {baseline_accuracy:.1f}%")
print(f"   Paper reports: 36.0% for Qwen2.5-Math-1.5B")


Loading MATH500 dataset...


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Loaded 500 problems
Subjects: ['Algebra', 'Counting & Probability', 'Geometry', 'Intermediate Algebra', 'Number Theory', 'Prealgebra', 'Precalculus']
Starting MATH500 baseline evaluation...
Tip: use math500.select(range(50)) for a quick 50-problem sanity check first


BASELINE — Before 1-Shot RLVR
Evaluating 10 problems | max_new_tokens=256



BASELINE — Before 1-Shot RLVR: 100%|██████████| 10/10 [05:02<00:00, 30.27s/it]


RESULTS: BASELINE — Before 1-Shot RLVR
  Overall accuracy : 1/10 = 10.0%
  (Paper baseline  : 36.0%  |  Paper post-1shot: 73.6%)

  Subject                          Ours   Paper base
  --------------------------------------------------
  Alg.                            33.3%        37.1%
  I.Alg.                           0.0%        43.3%
  N.T.                             0.0%        24.2%
  Prealg.                          0.0%        36.6%
  Precal.                          0.0%        33.9%

📊 Baseline MATH500 accuracy: 10.0%
   Paper reports: 36.0% for Qwen2.5-Math-1.5B


## Cell 10: Setup LoRA
LoRA lets us train only a small fraction of parameters — critical for T4 memory.

In [21]:
# ── LoRA configuration ─────────────────────────────────────────────────────
# We only train LoRA adapters, not full weights
# This reduces trainable params from ~1.5B to ~20M

lora_config = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    lora_dropout   = CFG['lora_dropout'],
    task_type      = TaskType.CAUSAL_LM,
    target_modules = CFG['lora_target'],
    bias           = 'none',
)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
# LoRA params will be added by GRPOTrainer automatically

print('✅ LoRA config ready')
print(f'   Rank (r)          : {CFG["lora_r"]}')
print(f'   Alpha             : {CFG["lora_alpha"]}')
print(f'   Target modules    : {CFG["lora_target"]}')
print(f'   Total model params: {total_params / 1e6:.1f}M')
print(f'   (LoRA trains only ~1-2% of these)')

✅ LoRA config ready
   Rank (r)          : 16
   Alpha             : 32
   Target modules    : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
   Total model params: 888.6M
   (LoRA trains only ~1-2% of these)


## Cell 11: GRPO Training Config

In [22]:
# ── GRPO Training Configuration ────────────────────────────────────────────
# Compare with paper settings in comments
grpo_config = GRPOConfig(
    output_dir              = CFG['output_dir'],
    run_name                = f'1shot_rlvr_4bit_{datetime.now().strftime("%Y%m%d_%H%M")}',
    max_steps               = CFG['num_train_steps'],
    save_steps              = CFG['save_steps'],
    logging_steps           = CFG['logging_steps'],
    per_device_train_batch_size = CFG['per_device_batch'],
    gradient_accumulation_steps = CFG['grad_accum'],
    num_generations         = CFG['num_generations'],
    max_completion_length   = CFG['max_new_tokens'],   # ← renamed
    max_prompt_length       = CFG['max_prompt_len'],
    temperature             = CFG['temperature'],
    learning_rate           = CFG['learning_rate'],
    optim                   = 'paged_adamw_8bit',
    lr_scheduler_type       = 'cosine',
    warmup_ratio            = 0.05,
    gradient_checkpointing  = True,
    bf16                    = False,
    fp16                    = True,
    beta                    = 0.001,
    report_to               = 'none',
    remove_unused_columns   = False,
    dataloader_num_workers  = 0,
)




print('✅ GRPO config ready')
print(f'   Effective batch size : {CFG["per_device_batch"] * CFG["grad_accum"]} (paper: 128)')
print(f'   Generations per step : {CFG["num_generations"]} (paper: 8)')
print(f'   Max response tokens  : {CFG["max_new_tokens"]} (paper: 3072)')
print(f'   Optimizer            : paged_adamw_8bit (memory efficient)')

✅ GRPO config ready
   Effective batch size : 2 (paper: 128)
   Generations per step : 2 (paper: 8)
   Max response tokens  : 128 (paper: 3072)
   Optimizer            : paged_adamw_8bit (memory efficient)


## Cell 12: Run 1-Shot RLVR Training
Core experiment — training on **π₁ only**, exactly like the paper.

In [23]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()  # ← this is the key fix

In [24]:
# After prepare_model_for_kbit_training
from peft import get_peft_model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should show ~1-5% trainable

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
# ── 1-Shot RLVR Training ───────────────────────────────────────────────────
# This is the main experiment from the paper
# Paper: trained on π₁ only, MATH500 improved 36% → 73.6%
# Our version: 4-bit quantized, smaller batch, fewer steps

print('=' * 60)
print('Starting 1-Shot RLVR Training (π₁ only)')
print('=' * 60)
print(f'Training example: {PI_1_PROMPT[:80]}...')
print(f'Ground truth    : {PI_1_ANSWER}')
print()

# Clear any cached memory
gc.collect()
torch.cuda.empty_cache()

mem_before_training = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory before training: {mem_before_training:.2f} GB')


# Initialize GRPOTrainer
trainer = GRPOTrainer(
    model            = model,        # already wrapped with LoRA
    args             = grpo_config,
    processing_class = tokenizer,
    train_dataset    = train_dataset_1shot,
    reward_funcs     = math_reward_fn,
    peft_config      = None,         # ← remove this, model already has LoRA
)

mem_after_trainer = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory after trainer init: {mem_after_trainer:.2f} GB')
print()
print('Starting training...')
print('(Expected: ~30-60 min on T4 for 200 steps)')
print()

# Train!
train_result = trainer.train()

print()
print('=' * 60)
print('✅ Training Complete!')
print(f'   Total steps    : {train_result.global_step}')
print(f'   Training loss  : {train_result.training_loss:.4f}')
print('=' * 60)

max_steps is given, it will override any value given in num_train_epochs


Starting 1-Shot RLVR Training (π₁ only)
Training example: The pressure P exerted by wind on a sail varies jointly as the area A of the sai...
Ground truth    : 12.8

GPU memory before training: 3.42 GB
GPU memory after trainer init: 3.42 GB

Starting training...
(Expected: ~30-60 min on T4 for 200 steps)



Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


## Cell 13: Post-Training Evaluation
Has the model improved? Compare with baseline.

In [16]:
# ── Evaluate AFTER 1-shot RLVR training ───────────────────────────────────

print('Running post-training evaluation...')
post_training_accuracy = evaluate_model(
    trainer.model, tokenizer, TEST_PROBLEMS, 'POST-TRAINING (After 1-Shot RLVR)'
)

print()
print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)
print(f'Before training : {baseline_accuracy:.1f}%')
print(f'After training  : {post_training_accuracy:.1f}%')
improvement = post_training_accuracy - baseline_accuracy
print(f'Improvement     : {improvement:+.1f}%')
print()
print('Paper comparison (on MATH500 with full precision):')
print('  Paper baseline         : 36.0%')
print('  Paper after 1-shot RLVR: 73.6%')
print('  Paper improvement      : +37.6%')
print()
if improvement > 0:
    print('✅ 1-Shot RLVR works even with 4-bit quantization on T4!')
else:
    print('⚠️  No improvement yet — try more training steps or check OOM issues')

Running post-training evaluation...

POST-TRAINING (After 1-Shot RLVR) Results:
  Accuracy: 0/8 = 0.0%
  ❌ GT=38         | Pred=           | Find the sum of all positive integers n such that n^2 - 19n ...
  ❌ GT=12         | Pred=           | How many integers between 1 and 200 are divisible by both 3 ...
  ❌ GT=14π        | Pred=28.0       | A circle has area 49π. What is its circumference? Let's thin...
  ❌ GT=3/2        | Pred=           | If log base 2 of x = 3, what is log base 4 of x? Let's think...
  ❌ GT=7          | Pred=           | What is the units digit of 7^2025? Let's think step by step ...
  ❌ GT=1/6        | Pred=           | Two dice are rolled. What is the probability both show the s...
  ❌ GT=4          | Pred=           | Find the number of real solutions to x^4 - 5x^2 + 4 = 0. Let...
  ❌ GT=384        | Pred=192        | A geometric sequence has first term 3 and common ratio 2. Wh...

RESULTS SUMMARY
Before training : 0.0%
After training  : 0.0%
Improvement     : 

In [18]:
# ── Post-training evaluation on MATH500 ───────────────────────────────────

print("Running post-training MATH500 evaluation...")

post_acc, post_subject_accs, post_results = evaluate_math500(
    trainer.model, tokenizer,
    dataset        = math500.select(range(10)),
    max_new_tokens = 256,
    desc           = "POST-TRAINING — After 1-Shot RLVR",
)

# ── Comparison table — mirrors paper Table 3 ──────────────────────────────
print(f"\n{'='*65}")
print("FINAL COMPARISON (mirrors paper Table 3)")
print(f"{'='*65}")
print(f"{'Subject':<28} {'Base':>8} {'Ours':>8} {'Δ':>8}  {'Paper π₁':>10}")
print(f"{'-'*65}")

paper_pi1 = {       # paper Table 3, {π₁} row
    'Algebra'               : 88.7,
    'Counting & Probability': 63.2,
    'Geometry'              : 56.1,
    'Intermediate Algebra'  : 62.9,
    'Number Theory'         : 79.0,
    'Prealgebra'            : 81.7,
    'Precalculus'           : 64.3,
}

for subj in sorted(baseline_subject_accs):
    base  = baseline_subject_accs[subj]
    post  = post_subject_accs.get(subj, 0.0)
    delta = post - base
    ref   = paper_pi1.get(subj, 0.0)
    ab    = {
        'Algebra':'Algebra','Counting & Probability':'C&P',
        'Geometry':'Geometry','Intermediate Algebra':'Int.Alg',
        'Number Theory':'Num.Th','Prealgebra':'Prealg',
        'Precalculus':'Precal'
    }.get(subj, subj[:8])
    print(f"  {ab:<26} {base:>7.1f}% {post:>7.1f}% {delta:>+7.1f}%  {ref:>9.1f}%")

print(f"{'-'*65}")
improvement = post_acc - baseline_acc
print(f"  {'MATH500 Overall':<26} {baseline_acc:>7.1f}% {post_acc:>7.1f}% {improvement:>+7.1f}%  {'73.6%':>9}")
print(f"{'='*65}")
print()
print("Paper context:")
print(f"  Paper baseline (full precision)  : 36.0%")
print(f"  Paper after 1-shot RLVR (2000 steps, 8xA100) : 73.6%")
print(f"  Our result ({CFG['num_train_steps']} steps, 1xT4, 4-bit QLoRA)  : {post_acc:.1f}%")


Running post-training MATH500 evaluation...

POST-TRAINING — After 1-Shot RLVR
Evaluating 10 problems | max_new_tokens=256



POST-TRAINING — After 1-Shot RLVR: 100%|██████████| 10/10 [04:25<00:00, 26.58s/it]


RESULTS: POST-TRAINING — After 1-Shot RLVR
  Overall accuracy : 1/10 = 10.0%
  (Paper baseline  : 36.0%  |  Paper post-1shot: 73.6%)

  Subject                          Ours   Paper base
  --------------------------------------------------
  Alg.                            33.3%        37.1%
  I.Alg.                           0.0%        43.3%
  N.T.                             0.0%        24.2%
  Prealg.                          0.0%        36.6%
  Precal.                          0.0%        33.9%


FINAL COMPARISON (mirrors paper Table 3)
Subject                          Base     Ours        Δ    Paper π₁
-----------------------------------------------------------------
  Algebra                       33.3%    33.3%    +0.0%       88.7%
  Int.Alg                        0.0%     0.0%    +0.0%       62.9%
  Num.Th                         0.0%     0.0%    +0.0%       79.0%
  Prealg                         0.0%     0.0%    +0.0%       81.7%
  Precal                         0.0%     0.0

## Cell 14: Test on Training Example π₁
Verify the model can solve the exact training problem.

In [14]:
# ── Test model on the training example π₁ ─────────────────────────────────
# Paper shows training accuracy saturates near 100% quickly
# Let's verify our model also solves it

def generate_response(model, tokenizer, prompt, max_new_tokens=512, temperature=0.6):
    """Generate a single response from the model."""
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=CFG['max_prompt_len'],
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = True,
            temperature    = temperature,
            pad_token_id   = tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print('Testing on training example π₁...')
print(f'Problem: {PI_1_PROMPT}')
print(f'Ground truth: {PI_1_ANSWER}')
print()

response = generate_response(trainer.model, tokenizer, PI_1_PROMPT)
extracted = extract_boxed_answer(response)
rewards   = math_reward_fn([response], [PI_1_ANSWER])

print('Model Response:')
print('-' * 50)
print(response[:1000])   # show first 1000 chars
if len(response) > 1000:
    print('... [truncated]')
print('-' * 50)
print(f'Extracted answer : {extracted}')
print(f'Ground truth     : {PI_1_ANSWER}')
print(f'Reward           : {rewards[0]}')
print(f'Correct?         : {"✅ YES" if rewards[0] == 1.0 else "❌ NO"}')

Testing on training example π₁...
Problem: The pressure P exerted by wind on a sail varies jointly as the area A of the sail and the cube of the wind's velocity V. When the velocity is 8 miles per hour, the pressure on a sail of 2 square feet is 4 pounds. Find the wind velocity when the pressure on 4 square feet of sail is 32 pounds. Let's think step by step and output the final answer within \boxed{}.
Ground truth: 12.8

Model Response:
--------------------------------------------------
 The pressure P exerted by wind on a sail is given by the equation \(P = k \cdot A \cdot V^3\), where \(k\) is a constant of proportionality, \(A\) is the area of the sail, and \(V\) is the wind's velocity. Given that when \(V = 8\) miles per hour, \(P = 4\) pounds, and \(A = 2\) square feet, we can use this information to find \(k\). 

First, we substitute the given values into the equation to solve for \(k\):

\[4 = k \cdot 2 \cdot 8^3\]

Next, we solve for \(k\):

\[4 = k \cdot 2 \cdot 512\]
\[4 = k

## Cell 16: Save Results

In [ ]:
# ── Save results and trained adapter ──────────────────────────────────────

os.makedirs(CFG['output_dir'], exist_ok=True)

# Save LoRA adapter (very small — only a few MB vs GB for full model)
adapter_path = os.path.join(CFG['output_dir'], 'lora_adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ LoRA adapter saved to {adapter_path}')

# Save experiment results to JSON
results = {
    'experiment'             : '1-Shot RLVR with 4-bit Quantization',
    'model'                  : CFG['model_name'],
    'quantization'           : f'4-bit {CFG["quant_type"]}',
    'lora_rank'              : CFG['lora_r'],
    'training_example'       : 'π₁ (wind pressure algebra)',
    'ground_truth'           : PI_1_ANSWER,
    'num_train_steps'        : CFG['num_train_steps'],
    'num_generations'        : CFG['num_generations'],
    'max_new_tokens'         : CFG['max_new_tokens'],
    'baseline_accuracy_pct'  : baseline_accuracy,
    'post_training_accuracy' : post_training_accuracy,
    'improvement_pct'        : post_training_accuracy - baseline_accuracy,
    'peak_vram_gb'           : round(peak_vram, 2),
    'total_vram_gb'          : round(total_vram, 2),
    'paper_baseline_pct'     : 36.0,
    'paper_post_training_pct': 73.6,
    'pi1_solve_rate_5samples': f'{correct_count}/5',
}

results_path = os.path.join(CFG['output_dir'], 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ Results saved to {results_path}')
print()
print('Final Results:')
for k, v in results.items():
    print(f'  {k:<35}: {v}')